In [ ]:
# ─── Configuration — set paths before running ───────────────────────────────
import sys, os
REPO_ROOT = os.path.abspath('../')   # DPsurv/ directory
sys.path.insert(0, REPO_ROOT)

# Path to frozen PANTHER prototype file (.pkl)
PROTO_PATH = ''   # e.g. 'data/splits/TCGA_LUAD_.../prototypes/prototypes_c16_....pkl'

# Example TCGA-LUAD slide to visualize
SLIDE_ID    = 'TCGA-93-8067-01Z-00-DX1.325d0cd5-9bb5-4f51-a197-75d3c3d19aaf'
SLIDE_PATH  = ''   # path to .svs whole-slide image
FEAT_H5_PATH = ''  # path to .h5 patch feature file for this slide

# Output directory for saved figures
OUT_DIR = 'outputs/LUAD'
os.makedirs(OUT_DIR, exist_ok=True)

IN_DIM = 1536   # UNI2 feature dimension
N_PROTO = 16    # number of GMM prototypes
print('Config OK. OUT_DIR:', OUT_DIR)

In [ ]:
import h5py
import openslide
import torch
import pandas as pd
from prototype_visualization_utils import (
    get_panther_encoder, visualize_categorical_heatmap, get_mixture_plot,
    get_default_cmap, visualize_risk_heatmap, visualize_thumbnail,
    get_mixture_plot_threshold,
)
from feature_extraction.tokenizer import PrototypeTokenizer


In [ ]:
### Loading PANTHER Encoder
panther_encoder = get_panther_encoder(in_dim=IN_DIM, p=N_PROTO, proto_path=PROTO_PATH)


In [ ]:
### open your WSI and features
slide_id = SLIDE_ID
slide_fpath = SLIDE_PATH
h5_feats_fpath = FEAT_H5_PATH

wsi = openslide.open_slide(slide_fpath)
h5 = h5py.File(h5_feats_fpath, 'r')

coords = h5['coords'][:].squeeze()
feats = torch.Tensor(h5['features'][:])
patch_size = h5['coords'].attrs['patch_size']*2

### get PANTHER representation and GMM mixtures
with torch.inference_mode():
    info = panther_encoder.representation(feats)
    qqs = info['qq']
    out = info['repr']
    tokenizer = PrototypeTokenizer(out_type='allcat',p=16)
    mus, pis, sigmas = tokenizer.forward(out)
    mus = mus[0].detach().cpu().numpy()
    qq = qqs[0,:,:,0].cpu().numpy()
    global_cluster_labels = qq.argmax(axis=1)

### Visualize the categorical heatmap and the GMM mixtures
cat_map = visualize_categorical_heatmap(
    wsi,
    coords, 
    global_cluster_labels, 
    label2color_dict=get_default_cmap(16),
    vis_level=wsi.get_best_level_for_downsample(128),
    patch_size=(patch_size, patch_size),
    alpha=1.0,
)

display(cat_map.resize((cat_map.width//4, cat_map.height//4)))
display(get_mixture_plot(mus))

In [ ]:
display(get_mixture_plot_threshold(mus, threshold=0.01, hide_xticks=True, ymax=0.4,figsize=(5.2, 1.5), bar_width=0.8))
mixture_fig = get_mixture_plot_threshold(mus, threshold=0.01, hide_xticks=True, ymax=0.4,figsize=(5.2, 1.5), bar_width=0.8)

In [ ]:
import os
out_dir = "OUT_DIR"
os.makedirs(out_dir, exist_ok=True)
barplot_save = os.path.join(out_dir, f"{slide_id}_Barplot_LUAD.png")
mixture_fig.savefig(barplot_save, dpi=300, bbox_inches="tight")

In [ ]:
original_map = visualize_thumbnail(wsi)
display(original_map.resize((original_map.width//4, original_map.height//4)))

In [ ]:
'''
import os
out_dir = "OUT_DIR"
os.makedirs(out_dir, exist_ok=True)
original_map_save = os.path.join(out_dir, f"{slide_id}_original_map_LUAD.png")
original_map.save(original_map_save)
'''

In [ ]:
risk_map = visualize_risk_heatmap(
    wsi,
    coords, 
    global_cluster_labels, 
    label2color_dict=get_default_cmap(16),
    vis_level=wsi.get_best_level_for_downsample(128),
    patch_size=(patch_size, patch_size),
    alpha=0.75,
    mu_components = [7.9308, 6.9184, 7.5312, 7.3336, 8.1886, 7.2684, 6.3103, 8.0469, 7.6380, 7.1896, 7.6489, 6.9156, 16.2268, 7.3263, 7.0673, 7.7975],
    pi_components = mus,
    )

display(risk_map.resize((risk_map.width//4, risk_map.height//4)))
'''
import os
out_dir = "OUT_DIR"
os.makedirs(out_dir, exist_ok=True)
cat_map_save = os.path.join(out_dir, f"{slide_id}_riskmap.png")
risk_map.save(cat_map_save)

sig = [0.2133, 0.1386, 0.0982, 0.4916, 0.1341, 0.3902, 0.2903, 0.4345, 0.2044, 0.5693, 0.2965, 0.3624, 0.2001, 0.6274, 0.2659, 0.5346]
hx = [2.7555, 0.9710, 0.8657, 1.1570, 9.3411, 1.6986, 2.7395, 4.0767, 1.8815, 2.9002, 1.1876, 2.9979, 2.0644, 4.0179, 0.9376, 0.8362]
'''

In [ ]:
'''
import os
from datetime import datetime
out_dir = "OUT_DIR"
os.makedirs(out_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# a) 保存热力图（建议同时保存原尺寸和缩略图）
cat_map_save_full = os.path.join(out_dir, f"{slide_id}_catmap_full_{ts}.png")
cat_map_save_small = os.path.join(out_dir, f"{slide_id}_catmap_quarter_{ts}.png")
cat_map.save(cat_map_save_full)
cat_map.resize((cat_map.width//4, cat_map.height//4)).save(cat_map_save_small)

# b) 保存 mixture 柱状图（matplotlib Figure）
mixture_save = os.path.join(out_dir, f"{slide_id}_gmm_mixtures_{ts}.png")
mixture_fig = get_mixture_plot(mus)
mixture_fig.savefig(mixture_save, dpi=300, bbox_inches="tight")

print(f"[Saved] {cat_map_save_full}")
print(f"[Saved] {cat_map_save_small}")
print(f"[Saved] {mixture_save}")
'''

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
def save_risk_colorbar_vfield(
    path,
    cmap_name='RdBu_r',
    ticks=(0.0, 0.5, 1.0),
    label=None,
    width_in=0.6, height_in=6.0, dpi=300,
    flip_vertical=False,         # True: 顶部显示 0
    white_bg=True,               # 设为 True 导出白底，无棋盘格
    show_outline=False           # 是否显示色条黑色外框
):
    cmap = mpl.cm.get_cmap(cmap_name)
    norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

    fig, ax = plt.subplots(figsize=(width_in, height_in), facecolor='white' if white_bg else None)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])

    cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=1.0, pad=0.1)
    cbar.set_ticks(list(ticks))
    if label is not None:
        cbar.set_label(label, rotation=0, labelpad=8)

    if flip_vertical:
        cbar.ax.invert_yaxis()
    if not show_outline:
        cbar.outline.set_visible(False)

    # 确保是白底导出（无透明）
    ax.remove()
    fig.savefig(path,
                dpi=dpi,
                bbox_inches='tight',
                transparent=False if white_bg else True,
                facecolor='white' if white_bg else None)
    plt.close(fig)
# 用法示例（与 visualize_risk_heatmap 默认一致：cmap='RdBu_r'）
save_risk_colorbar_vfield(
    path='OUT_DIR/risk_colorbar_RdBu_r.png',
    cmap_name='RdBu_r',
    ticks=(0.0, 0.5, 1.0),
    label=None,              # 或 "risk (v)"
    flip_vertical=False      # False: 底部=0, 顶部=1；RdBu_r 下 0=红, 1=蓝
)

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def _normalize_mu(mu, keep_mask, stretch='percentile', p_low=5, p_high=95):
    mu = np.asarray(mu, float)
    if not np.any(keep_mask):
        lo, hi = np.nanmin(mu), np.nanmax(mu)
    else:
        if stretch == 'percentile':
            lo, hi = np.percentile(mu[keep_mask], [p_low, p_high])
        else:
            lo, hi = np.nanmin(mu[keep_mask]), np.nanmax(mu[keep_mask])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.full_like(mu, 0.5, dtype=float), (lo, hi)
    mu_norm = np.clip((mu - lo) / (hi - lo), 0, 1).astype(float)
    return mu_norm, (lo, hi)

def _brighten_endpoints(base='RdBu_r', left='#FF0000', right='#0033FF', n=256):
    cm = mpl.cm.get_cmap(base, n)
    tab = cm(np.linspace(0, 1, n))
    from matplotlib.colors import to_rgb
    tab[0, :3]  = to_rgb(left)   # v=0 端（红）
    tab[-1, :3] = to_rgb(right)  # v=1 端（蓝）
    return mpl.colors.ListedColormap(tab, name=f'{base}-bright')

def get_mixture_plot_risk(
    mu_components,                 # 每簇 μ
    pi_components=None,            # 每簇 π，用于过滤
    pi_thresh=0.01,                # π<thresh 删掉
    cmap_name='RdBu_r',
    high_is='red',                 # 高 risk=红（推荐）
    stretch='percentile', p_low=5, p_high=95,
    gamma=1.0,                     # 与热图完全一致建议=1.0
    brighten=False, left='#FF0000', right='#0033FF',
    # —— 与 threshold 版对齐的外观参数 —— #
    hide_xticks=True, ymax=1.0, figsize=(4, 2.5), bar_width=0.6,
    tick_labelsize=10, y_label='Risk Score',  # 你之前的命名
    font_path="./Arial.ttf",
    min_bar=1e-3                   # 最小可视柱高
):
    mu = np.asarray(mu_components, float).reshape(-1)
    pi = np.ones_like(mu, float) if pi_components is None else np.asarray(pi_components, float).reshape(-1)
    if pi.shape != mu.shape:
        raise ValueError(f"pi_components shape {pi.shape} != mu shape {mu.shape}")

    # 过滤
    keep = np.isfinite(mu) & np.isfinite(pi) & (pi >= pi_thresh)
    idx  = np.where(keep)[0]
    if idx.size == 0:
        fig, ax = plt.subplots(figsize=figsize, dpi=300)
        ax.text(0.5, 0.5, f'No clusters with pi ≥ {pi_thresh}', ha='center', va='center')
        ax.axis('off'); plt.close(fig); return fig

    mu_keep = mu[idx]

    # μ → μ_norm；risk = 1 - μ_norm
    mu_norm, _ = _normalize_mu(mu_keep, keep_mask=np.ones_like(mu_keep, bool),
                               stretch=stretch, p_low=p_low, p_high=p_high)
    risk = 1.0 - mu_norm                       # 柱高
    risk_for_color = np.clip(risk, 0, 1) ** float(gamma)  # 仅用于颜色

    # 颜色映射（RdBu_r: v=0红 v=1蓝；高risk更红 ⇒ v=1-risk? 不，是 1-红=蓝，所以：）
    v = 1.0 - risk_for_color if high_is == 'red' else risk_for_color
    cmap = _brighten_endpoints(cmap_name, left, right) if brighten else mpl.cm.get_cmap(cmap_name)
    colors = cmap(np.clip(v, 0, 1))

    # 按原编号顺序（和 threshold 版一致）
    order = np.argsort(idx)
    risk_plot   = risk[order]
    colors_plot = colors[order]

    # 画图（外观对齐 threshold 版）
    prop = fm.FontProperties(fname=font_path)
    mpl.rcParams['axes.linewidth'] = 1.0
    mpl.rcParams['pdf.fonttype'] = 42
    mpl.rcParams['ps.fonttype']  = 42

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    x = np.arange(len(risk_plot))
    heights = np.clip(np.maximum(risk_plot, float(min_bar)), 0.0, None)

    ax.bar(x, heights, color=colors_plot, edgecolor='none', width=bar_width)

    # x 轴与标签（与 threshold 版一致）
    if hide_xticks:
        ax.set_xticks([]); ax.set_xlabel('')
        ax.tick_params(axis='x', length=0)
    else:
        ax.set_xticks(x)
        ax.set_xticklabels([f'c{i}' for i in idx], fontproperties=prop, fontsize=tick_labelsize)
        ax.set_xlabel('Cluster', fontproperties=prop, fontsize=10)

    # y 轴
    ax.set_ylabel(y_label, fontproperties=prop, fontsize=10)
    ax.tick_params(labelsize=tick_labelsize)
    if ymax is None:
        ymax = float(risk_plot.max()) * 1.05 if len(risk_plot) else 1.0
    ax.set_ylim(0, ymax)

    plt.tight_layout(); plt.close(fig)
    return fig


fig = get_mixture_plot_risk(
    mu_components=[7.9308, 6.9184, 7.5312, 7.3336, 8.1886, 7.2684, 6.3103, 8.0469,
                   7.6380, 7.1896, 7.6489, 6.9156, 16.2268, 7.3263, 7.0673, 7.7975],
    pi_components=mus,     # ← 确保这是权重数组，长度=16
    cmap_name='RdBu_r',
    high_is='blue',
    pi_thresh=0.01,
    stretch='percentile', p_low=5, p_high=95,
    gamma=1.0,
    brighten=False,
    sort='index',
    figsize=(6,3), dpi=300,
)
#fig.savefig("OUT_DIR/cluster_risk_bars.png", dpi=300)


In [ ]:
fig = get_mixture_plot_risk(
    mu_components=[7.9308, 6.9184, 7.5312, 7.3336, 8.1886, 7.2684, 6.3103, 8.0469,
                   7.6380, 7.1896, 7.6489, 6.9156, 16.2268, 7.3263, 7.0673, 7.7975],
    pi_components=mus,
    pi_thresh=0.01,
    cmap_name='RdBu_r', high_is='blue',
    stretch='percentile', p_low=5, p_high=95,
    gamma=1.0, brighten=False,            # 若要和热图严格一致
    hide_xticks=True, ymax=1.0, figsize=(6,3.5), bar_width=0.6,
    min_bar=3e-2
)
display(fig)
fig.savefig("OUT_DIR/cluster_risk_bars.png", dpi=300, bbox_inches="tight")
